# 🌍 Azure AI Translator — Lab AI-102

**Objectif**: Traduire et détecter des langues avec Azure AI Translator.

## Compétences AI-102 couvertes
- Traduction vers plusieurs langues simultanément
- Détection automatique de la langue source
- Translittération (changement de script)
- Recherche dans le dictionnaire bilingue
- Lister les langues supportées

In [ ]:
%pip install requests python-dotenv -q

In [ ]:
import os
import uuid
import requests
from dotenv import load_dotenv

load_dotenv('../.env')

ENDPOINT = os.getenv('AZURE_TRANSLATOR_ENDPOINT', 'https://api.cognitive.microsofttranslator.com')
KEY = os.getenv('AZURE_TRANSLATOR_KEY')
REGION = os.getenv('AZURE_TRANSLATOR_REGION', 'eastus')
API_VERSION = '3.0'

def get_headers():
    return {
        'Ocp-Apim-Subscription-Key': KEY,
        'Ocp-Apim-Subscription-Region': REGION,
        'Content-Type': 'application/json',
        'X-ClientTraceId': str(uuid.uuid4())
    }

print('✅ Configuration Translator chargée')

## 1. Traduction simple et multi-langues

In [ ]:
# AI-102: Un seul appel API peut traduire vers plusieurs langues simultanément

text_fr = "Azure AI Foundry simplifie le développement d'applications d'intelligence artificielle."

# Traduction vers 5 langues en un seul appel
response = requests.post(
    f"{ENDPOINT}/translate",
    params={'api-version': API_VERSION, 'to': ['en', 'es', 'ar', 'zh-Hans', 'de']},
    headers=get_headers(),
    json=[{'text': text_fr}]
)
results = response.json()

result = results[0]
detected = result.get('detectedLanguage', {})
print(f"Langue source détectée: {detected.get('language')} ({detected.get('score', 0):.0%})")
print(f"\nTexte original (FR): {text_fr}\n")
print("Traductions:")
for t in result['translations']:
    print(f"  [{t['to'].upper()}] {t['text']}")

## 2. Détection de langue

In [ ]:
# AI-102: L'API Detect retourne la langue avec score et alternatives

texts_to_detect = [
    "This is a legal contract between two parties.",
    "Dies ist ein Vertrag zwischen zwei Parteien.",
    "هذا عقد بين طرفين.",
    "これは二者間の契約です。",
]

response = requests.post(
    f"{ENDPOINT}/detect",
    params={'api-version': API_VERSION},
    headers=get_headers(),
    json=[{'text': t} for t in texts_to_detect]
)

results = response.json()
for text, result in zip(texts_to_detect, results):
    print(f"Texte: '{text[:50]}...'")
    print(f"  Langue: {result['language']} (confiance: {result['score']:.0%})")
    print(f"  Traduction supportée: {result['isTranslationSupported']}")
    print()

## 3. Translittération

In [ ]:
# AI-102: Translittération = changer de script sans changer de langue
# Utile pour l'arabe, le japonais, le chinois → script latin

examples = [
    {'text': 'مرحبا', 'language': 'ar', 'fromScript': 'Arab', 'toScript': 'Latn'},
    {'text': 'こんにちは', 'language': 'ja', 'fromScript': 'Jpan', 'toScript': 'Latn'},
]

for ex in examples:
    response = requests.post(
        f"{ENDPOINT}/transliterate",
        params={
            'api-version': API_VERSION,
            'language': ex['language'],
            'fromScript': ex['fromScript'],
            'toScript': ex['toScript']
        },
        headers=get_headers(),
        json=[{'text': ex['text']}]
    )
    result = response.json()[0]
    print(f"  '{ex['text']}' → '{result['text']}' (script: {result['script']})")

## 4. Dictionnaire bilingue

In [ ]:
# AI-102: Dictionary lookup retourne les traductions alternatives avec POS et rétro-traduction

response = requests.post(
    f"{ENDPOINT}/dictionary/lookup",
    params={'api-version': API_VERSION, 'from': 'fr', 'to': 'en'},
    headers=get_headers(),
    json=[{'text': 'contrat'}]
)

result = response.json()[0]
print(f"Traductions du mot 'contrat' (FR→EN):")
for t in result['translations'][:5]:
    back = ', '.join([b['displayText'] for b in t['backTranslations'][:3]])
    print(f"  {t['displayTarget']} [{t['posTag']}] (confiance: {t['confidence']:.0%}) → {back}")

## 5. Lister les langues supportées

In [ ]:
# AI-102: Vérifier quelles langues sont supportées pour traduction/translittération

response = requests.get(
    f"{ENDPOINT}/languages",
    params={'api-version': API_VERSION}
)
data = response.json()

translation_langs = data.get('translation', {})
print(f"Langues de traduction supportées: {len(translation_langs)}")

# Afficher quelques langues importantes
important = ['fr', 'en', 'es', 'de', 'ar', 'zh-Hans', 'pt', 'ru', 'ja', 'ko']
print("\nLangues clés:")
for code in important:
    if code in translation_langs:
        lang = translation_langs[code]
        print(f"  {code}: {lang['name']} ({lang['nativeName']})")